# TriaCV — Sprint 1 : Exploration des Données (EDA) & Prétraitement NLP

Ce notebook présente l'analyse exploratoire complète du jeu de données Kaggle **Resume Dataset** (25 catégories professionnelles) et illustre chaque étape du pipeline de nettoyage et normalisation de texte.

In [ ]:
import os
import sys
import json
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ajout du dossier racine au sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.utils.dataset_loader import load_or_create_dataset, CATEGORIES_INFO
from src.preprocessing.cleaner import ResumeCleaner
from src.preprocessing.extractor import ResumeExtractor

sns.set_theme(style='whitegrid')
print('Modules importés avec succès.')

## 1. Chargement du Jeu de Données Brut

In [ ]:
df_raw = load_or_create_dataset('../data/raw')
print(f'Dimensions du dataset brut : {df_raw.shape}')
print(f'Nombre de catégories uniques : {df_raw["Category"].nunique()}')
df_raw.head(5)

## 2. Analyse de la Distribution des Catégories

In [ ]:
plt.figure(figsize=(14, 8))
cat_counts = df_raw['Category'].value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, palette='viridis')
plt.title('Distribution des CVs par Catégorie Professionnelle', fontsize=14, fontweight='bold')
plt.xlabel('Nombre de CVs')
plt.ylabel('Catégorie')
plt.show()

## 3. Application du Pipeline de Nettoyage NLP

Le pipeline `ResumeCleaner` effectue :
1. Normalisation Unicode (accents, caractères spéciaux)
2. Suppression des URLs, emails, téléphones, hashtags et mentions
3. Protection des mots-clés techniques clés (`c++`, `.net`, `c#`, `react.js`, `node.js`)
4. Suppression de la ponctuation et caractères de contrôle
5. Conversion en minuscules
6. Suppression des stopwords et lemmatisation WordNet

In [ ]:
cleaner = ResumeCleaner(remove_stopwords=True, lemmatize=True)

# Exemple sur un CV brut
sample_raw = df_raw.iloc[0]['Resume']
sample_clean = cleaner.clean_text(sample_raw)

print('--- TEXTE BRUT (Extrait) ---')
print(sample_raw[:300])
print('\n--- TEXTE NETTOYÉ & LEMMATISÉ (Extrait) ---')
print(sample_clean[:300])

## 4. Traitement Global et Sauvegarde du Dataset Nettoyé

In [ ]:
df_raw['Clean_Resume'] = df_raw['Resume'].apply(cleaner.clean_text)
df_raw['Clean_Word_Count'] = df_raw['Clean_Resume'].apply(lambda x: len(x.split()))

# Sauvegarde
os.makedirs('../data/processed', exist_ok=True)
df_raw.to_csv('../data/processed/clean_resumes.csv', index=False)
print(f'Jeu de données nettoyé sauvegardé ({len(df_raw)} enregistrements).')